# Emotion Prediction from Text (6 Emotions)

**Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Course:** Machine Learning Fundamentals (AIC354), Assignment 4, Task 1 (Multi-class)  
**Instructor:** Dr. Rao Muhammad Adeel Nawab  
**University:** COMSATS University Islamabad, Lahore Campus  

**Deployment Link:** [https://spam-detector-hassan.streamlit.app](https://spam-detector-hassan.streamlit.app)  
**GitHub:** [https://github.com/MHassan0000/ML-Prediction-Models](https://github.com/MHassan0000/ML-Prediction-Models)

In this notebook I build a multi-class emotion classifier that predicts one of 6 emotions from short text: sadness, joy, love, anger, fear and surprise. I use TF-IDF for feature extraction and compare 6 ML algorithms using 10-fold cross validation.

**6 Algorithms compared:**
1. Random Forest
2. Multinomial Naive Bayes
3. k-Nearest Neighbors
4. Linear SVC
5. Logistic Regression
6. Decision Tree

## Step 1: Import Libraries

In [ ]:
import os
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from tqdm import tqdm

pd.set_option('display.float_format', lambda v: f'{v:.4f}')

print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Scikit-learn : {sklearn.__version__}')
print(f'Matplotlib   : {matplotlib.__version__}')

## Step 2: Load Dataset

Using all 20,000 labelled texts from the dair-ai/emotion dataset with all 6 emotion classes.

In [ ]:
DATA_FILE = 'emotion_full.csv'

if not os.path.isfile(DATA_FILE):
    try:
        from google.colab import files as colab_files
        print('Upload emotion_full.csv when the dialog appears...')
        colab_files.upload()
    except ImportError:
        raise FileNotFoundError(f'{DATA_FILE} not found. Put it in the same folder as this notebook.')
else:
    print(f'Found {DATA_FILE}')

df = pd.read_csv(DATA_FILE)

emotion_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
df['emotion'] = df['label'].map(emotion_map)

print(f'Loaded {len(df):,} texts with {df["label"].nunique()} emotion classes')
df.head(10)

## Step 3: Understand the Data

In [ ]:
print('Shape:', df.shape)
print(f'Missing values: {df.isnull().sum().sum()}')
print('\nEmotion distribution:')
print(df['emotion'].value_counts())

In [ ]:
print('\nOne sample text per emotion:')
print('=' * 70)
for emo in emotion_map.values():
    sample = df[df['emotion'] == emo].iloc[0]['text']
    print(f'  {emo.upper():>8}:  "{sample[:75]}"')

In [ ]:
plt.figure(figsize=(10, 5))
counts = df['emotion'].value_counts()
colors = ['#4A90D9', '#F5A623', '#D0021B', '#7B68EE', '#50C878', '#FF69B4']
bars = plt.bar(counts.index, counts.values, color=colors)
for bar, v in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
             f'{v:,}', ha='center', fontweight='bold')
plt.title('Emotion Distribution in Dataset')
plt.xlabel('Emotion')
plt.ylabel('Number of Texts')
plt.tight_layout()
plt.show()

## Step 4: Pre-processing

Lowercase, remove special characters, collapse extra whitespace.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

print('Before and after cleaning (first 3 rows):')
for i in range(3):
    print(f'  BEFORE: "{df.iloc[i]["text"]}"')
    print(f'  AFTER : "{df.iloc[i]["clean_text"]}"')
    print()

print(f'Cleaned {len(df):,} texts.')

## Step 5: Feature Extraction

TF-IDF with word unigrams, max 5000 features. I fit it on all data here (for preview) and also inside each fold during cross validation.

In [ ]:
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 1), max_features=5000)

X = tfidf.fit_transform(df['clean_text'])
y = df['label'].values

print(f'Feature matrix shape : {X.shape}')
print(f'Vocabulary size      : {len(tfidf.vocabulary_):,} words')
print(f'Target labels        : {sorted(np.unique(y))}')

## Step 6: Define 6 ML Algorithms

In [ ]:
algorithms = [
    ('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('MultinomialNB',       MultinomialNB()),
    ('k-NN',                KNeighborsClassifier(n_neighbors=5)),
    ('LinearSVC',           LinearSVC(max_iter=2000, random_state=42)),
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Decision Tree',       DecisionTreeClassifier(random_state=42)),
]

print(f'{len(algorithms)} algorithms defined:')
for i, (name, _) in enumerate(algorithms, 1):
    print(f'  {i}. {name}')

## Step 7: Set Up 10-Fold Cross Validation

In [ ]:
K = 10
kfold = KFold(n_splits=K, shuffle=True, random_state=42)

print(f'K = {K} folds | Total samples: {len(X.toarray()):,}')
print(f"\n{'Fold':<6} {'Train':>10} {'Test':>8}")
for i, (tr, te) in enumerate(kfold.split(X), 1):
    print(f'{i:<6} {len(tr):>10,} {len(te):>8,}')

## Step 8: Run Cross Validation

In [ ]:
results = {}
total_runs = len(algorithms) * K

with tqdm(total=total_runs, desc='Cross Validation', unit='fold') as pbar:
    for algo_name, algo_model in algorithms:
        fold_accs = []
        for tr_idx, te_idx in kfold.split(X):
            m = clone(algo_model)
            m.fit(X[tr_idx], y[tr_idx])
            acc = accuracy_score(y[te_idx], m.predict(X[te_idx]))
            fold_accs.append(acc)
            pbar.update(1)
        results[algo_name] = fold_accs
        print(f'  {algo_name:<22} -> Mean Accuracy: {np.mean(fold_accs):.4f}')

print('\nCross validation complete!')

## Step 9: Compare Results

In [ ]:
results_df = pd.DataFrame(results, index=[f'Fold {i}' for i in range(1, K + 1)])
results_df.loc['Mean'] = results_df.mean()
results_df.loc['Std']  = results_df.iloc[:K].std()
print('Accuracy of each algorithm on each fold:')
print(results_df.to_string())

In [ ]:
summary_rows = []
for name in results:
    summary_rows.append({
        'Algorithm':     name,
        'Mean Accuracy': np.mean(results[name]),
        'Std Deviation': np.std(results[name]),
        'Min Accuracy':  np.min(results[name]),
        'Max Accuracy':  np.max(results[name]),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Algorithm')
summary_df = summary_df.sort_values('Mean Accuracy', ascending=False)
print('Algorithm comparison sorted by mean accuracy:')
summary_df

In [ ]:
algo_names = list(results.keys())
bar_colors = ['#4A90D9', '#F5A623', '#D0021B', '#7B68EE', '#50C878', '#FF69B4']
mean_accs  = [np.mean(results[n]) for n in algo_names]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

bars = axes[0].bar(algo_names, mean_accs, color=bar_colors)
for bar, acc in zip(bars, mean_accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f'{acc:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Mean Accuracy per Algorithm')
axes[0].set_ylabel('Mean Accuracy')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=30)

bp = axes[1].boxplot([results[n] for n in algo_names], labels=algo_names, patch_artist=True)
for patch, c in zip(bp['boxes'], bar_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
axes[1].set_title('Accuracy Distribution')
axes[1].set_ylabel('Accuracy')
axes[1].tick_params(axis='x', rotation=30)

for i, name in enumerate(algo_names):
    axes[2].plot(range(1, K + 1), results[name], marker='o', label=name, color=bar_colors[i])
axes[2].set_title('Accuracy per Fold')
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=8)
axes[2].set_xticks(range(1, K + 1))

plt.tight_layout()
plt.show()

## Step 10: Observations

In [ ]:
best_algo  = max(results, key=lambda k: np.mean(results[k]))
worst_algo = min(results, key=lambda k: np.mean(results[k]))

print(f'Best  algorithm: {best_algo}  (Mean Accuracy: {np.mean(results[best_algo]):.4f})')
print(f'Worst algorithm: {worst_algo} (Mean Accuracy: {np.mean(results[worst_algo]):.4f})')

best_model_obj = dict(algorithms)[best_algo]
best_cm_model  = clone(best_model_obj)

for tr, te in kfold.split(X):
    pass

best_cm_model.fit(X[tr], y[tr])
y_pred_cm = best_cm_model.predict(X[te])

cm = confusion_matrix(y[te], y_pred_cm)
emotion_names = [emotion_map[i] for i in range(6)]
cm_df = pd.DataFrame(cm, index=emotion_names, columns=emotion_names)
cm_df.index.name   = 'Actual'
cm_df.columns.name = 'Predicted'
print(f'\nConfusion Matrix for {best_algo} (last fold):')
print(cm_df)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=range(6), yticks=range(6),
       xticklabels=emotion_names, yticklabels=emotion_names,
       ylabel='Actual', xlabel='Predicted',
       title=f'Confusion Matrix: {best_algo}')
for i in range(6):
    for j in range(6):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()

## Step 11: Train Final Model

In [ ]:
print(f'Training final model: {best_algo}')

final_model = clone(dict(algorithms)[best_algo])
final_model.fit(X, y)

with open('emotion_multiclass_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
with open('emotion_multiclass_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print(f'Trained on {X.shape[0]:,} texts.')
print('Saved emotion_multiclass_model.pkl and emotion_multiclass_tfidf.pkl')

## Step 12: Test Prediction

In [ ]:
with open('emotion_multiclass_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('emotion_multiclass_tfidf.pkl', 'rb') as f:
    loaded_tfidf = pickle.load(f)

example_texts = [
    'I am so happy today, everything is going great!',
    'I feel really sad and lonely right now.',
    'This makes me so angry, I cannot believe it happened.',
    'I am terrified of what might happen next.',
    'I love you so much, you mean the world to me.',
    'Wow, I did not expect that at all! What a surprise!',
    'I feel empty and hopeless about the future.',
    'She surprised me with the most beautiful gift.',
]

print('Predictions on example texts:')
print('=' * 65)
for text in example_texts:
    clean = re.sub(r'\s+', ' ', re.sub(r'[^a-z\s]', '', text.lower())).strip()
    pred  = loaded_model.predict(loaded_tfidf.transform([clean]))[0]
    print(f'  Text    : "{text}"')
    print(f'  Emotion : {emotion_map[pred]}')
    print()

print('Classification Report (last fold test set):')
print(classification_report(y[te], y_pred_cm, target_names=emotion_names))